## Literature Ingestion and Processing Pipeline

Lead     : `Aditya Ghosh / adityaghosh5626-creator`

Issue    : [Github Issue #103](https://github.com/petadex/igem-toronto/issues/103) — _Literature Scraping and Expansion_

Start    : `YYYY-MM-DD`

Complete : `YYYY-MM-DD`

Files    : `~/igem-toronto/files/YYMMDD_issue103_<slug>/` — local working directory (active analysis, intermediate outputs)

S3 files :
- `s3://plastic-literature-activity-data/docling_sectioned_text_jsons/`
- `s3://plastic-literature-activity-data/original_pdfs/`

---

### Data Accessed: YYYY-MM-DD
```bash
# commands used to pull input data
aws s3 cp s3://petadex/original_pdfs/ ./
```

## Introduction

To populate the PETra database with relevant plastic-degrading enzyme data, an automated ingestion pipeline was engineered to systematically discover, filter and extract academic literature. This workflow leverages a cyclical expansion model, utilizing an initial seed set of literature to map forward and backward citations, exponentially increasing the candidate pool. The pipeline addresses the challenge of identifying high-yield experimental data by routing tens of thousands of candidate papers through a relevance triage funnel before executing targeted full-text retrieval using institutional and open-access scraping protocols.

## Objectives

1. Acquire initial DOIs [~11,600] for papers using the PAZy API, PETadex NR accessions and broad keyword web queries relating to papers with activity on proteins that degrade plastics.
2. Screen and filter metadata to isolate a core set of [~900] highly relevant seed papers.
3. Expand the literature map by acquiring the references and citing articles for the [900] seeds, generating a secondary pool of [~50,000] papers.
4. Retrieve full-text documents for the final filtered set of [~4,600] papers using Selenium, Scihub and UofT library access, ultimately converting them into structured JSONS.

---

## Materials and Methods

### System Initialization

The compute environment relies on a custom Selenium and Chromium scraper setup for full-text acquisition. Network access is routed to utilize Scihub and UofT libraries for bypassing paywalls.

### Data Initialization

Initial DOIs and accessions are compiled via the PAZy API, PETadex NR database and targeted database article searches.

```bash
# Accessed: YYYY-MM-DD
# <command used to generate or download input data>
```

### Processing Steps

**Step 1: Input Streams (Acquiring DOIs)**
- Pull [~6,000] DOIs from PAZy API and PETadex NR accessions, alongside [~5,600] DOIs from broad keyword web queries, establishing an initial pool of [~11,600] papers.

**Step 2: Primary Relevance Triage**
- Route [~11,600] of these initial papers into the scraping and screening funnel. Acquire metadata (titles, authors, abstracts) and filter the papers based on relevance to plastic degrading enzymes to yield [~900] core seed papers.

**Step 3: Article Expansion**
- Process the [~900] seed papers to acquire DOIs of articles they reference, as well as DOIs of new articles that cite them. This citation mapping expands the pool to [~50,000] candidate papers.

**Step 4: Secondary Relevance Triage**
- Feed the expanded pool of [~50,000] papers back into the relevance triage funnel for secondary screening against the plastic-degrading enzyme criteria. This isolates a final highly relevant target list of [~4,600] papers.

**Step 5: Article Retrieval**
- Execute the Selenium and Chromium scraper, leveraging Scihub and UofT libraries, to acquire the full text for the [~4,600] filtered papers.

**Step 6: Synthesis**
- Convert the retrieved corpus of [~4,600] full-text articles into a final set of structured JSONS.

```bash
# Uploaded: YYYY-MM-DD
aws s3 cp structured_literature_corpus.json s3://petadex/docling_sectioned_text_jsons/
```

## Results & Discussion

![AZA ingestion pipeline](../resources/260814_issue90_Scraping%20Activity%20Data%20from%20Papers/AZA%20ingestion%20pipeline.svg)

The cyclical ingestion pipeline successfully processed multiple data streams to identify and extract a highly refined dataset. By utilizing a two-pass triage system, the pipeline expanded a small core of relevant literature into a vast candidate pool before aggressively filtering it down to the most critical articles.

**Key metric**: ~4,600 / ~50,000 expanded items (~9.2%) _met the secondary screening criteria for full-text retrieval_.

**Breakdown by category**:
- Input Streams: ~11,600 papers total (~6k from PAZY/PETadex NR, ~5.6k from broad keyword queries).
- Pre-Expansion Seeds: ~900 papers passing the primary triage.
- Post-Expansion Candidates: ~50,000 papers generated via citation mapping.
- Final Retrieved Corpus: ~4,600 papers converted into JSONS.

**Observations**:
- The expansion mechanics are highly effective; mapping the references and citing articles of just ~900 seed papers yielded ~50,000 new candidate DOIs.
- The triage funnel acts as a critical bottleneck, aggressively filtering the ~50,000 expanded candidates down to ~4,600 to ensure compute resources are only spent retrieving relevant full texts.
- Full-text retrieval relies heavily on a diversified access strategy, combining automated Chromium scraping with Scihub and UofT library authentication to maximize the yield of the final ~4,600 papers.

**Follow-up questions**:
- What is the failure rate of the Selenium scraper when navigating UofT library proxies compared to Scihub?
- Of the ~1,000 papers dropped between the initial ~11,600 input stream and the ~10,600 entering the first triage funnel, were these duplicates or invalid DOIs?
- Can we further optimize the metadata screening criteria to reduce the computational load during the secondary triage of the ~50,000 expanded papers?
